# Tier 0 graph: first overview

Explore the pinned Translator KG release `2026_06_21` before preparing ULTRA inputs.
This notebook uses **Python's standard library**, the downloaded release metadata,
and a streaming archive inventory. It never loads the full graph into memory or extracts it.
Run with a Python 3 Jupyter kernel (the archive inventory also needs `zstd` on PATH).

**Scope:** counts below are publisher-reported metadata, not independently counted
records. They do not establish unique drug–disease pairs, connectivity, duplicate
rates, or prediction quality. No model is run here.


In [1]:
from pathlib import Path
from collections import Counter
import csv
import json
import shutil
import subprocess
import tarfile

# Works when launched from this notebook folder, the experiment, or the repository.
ROOT = next((p / "experiments" / "ultra" if (p / "experiments" / "ultra" / "download.json").exists() else p
             for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "experiments" / "ultra" / "download.json").exists()
             or (p / "download.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Launch from inside the Shepherd repository.")
manifest = json.loads((ROOT / "download.json").read_text())
files = {Path(item["path"]).name: item for item in manifest["files"]}
metadata = json.loads((ROOT / files["graph-metadata.json"]["path"]).read_text())
archive = ROOT / files["translator_kg.tar.zst"]["path"]
OUTPUT = ROOT / "output" / "overview"
OUTPUT.mkdir(parents=True, exist_ok=True)
assert metadata["version"] == manifest["release"]
schema = metadata["schema"]
nodes = schema["nodes_summary"]
edges = schema["edges_summary"]

def table(rows, columns, limit=15):
    rows = list(rows)
    shown = rows[:limit]
    values = [[str(row.get(key, "")) for key in columns] for row in shown]
    widths = [max([len(key)] + [len(row[i]) for row in values]) for i, key in enumerate(columns)]
    print(" | ".join(key.ljust(widths[i]) for i, key in enumerate(columns)))
    print("-+-".join("-" * width for width in widths))
    for row in values:
        print(" | ".join(value.ljust(widths[i]) for i, value in enumerate(row)))
    if len(rows) > limit:
        print(f"Showing {limit} of {len(rows)} rows; full table exported.")

def export(name, rows, columns):
    with (OUTPUT / name).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

def counts_table(counts, name, denominator=None):
    rows = [{"label": label, "count": count,
             **({"percent": round(100 * count / denominator, 2)} if denominator else {})}
            for label, count in sorted(counts.items(), key=lambda item: item[1], reverse=True)]
    columns = ["label", "count"] + (["percent"] if denominator else [])
    export(name, rows, columns)
    table(rows, columns)
    return rows

print(f"Dataset: {metadata['name']} / {metadata['version']}")
print(f"Biolink: {metadata.get('biolinkVersion')} | Babel: {metadata.get('babelVersion')}")
print(f"Reported nodes: {nodes['total_count']:,}")
print(f"Reported edges: {edges['total_count']:,}")
print(f"Predicates: {len(edges['predicates'])} | Primary-source labels: {len(edges['primary_knowledge_sources'])}")
print(f"Archive: {archive.stat().st_size / 1e9:.2f} GB compressed")
print(f"Output directory: {OUTPUT}")


Dataset: translator_kg / 2026_06_21
Biolink: 4.4.2 | Babel: 2025sep1
Reported nodes: 1,754,754
Reported edges: 28,709,074
Predicates: 60 | Primary-source labels: 35
Archive: 3.23 GB compressed
Output directory: /home/agonzalez/shepherd/experiments/ultra/output/overview


## 1. Check metadata consistency

Compare the detailed category/predicate groups with the release totals. A match
checks the metadata's internal consistency, not the raw archive contents.


In [2]:
checks = {
    "node category-group total": sum(row["count"] for row in schema["nodes"]) == nodes["total_count"],
    "node prefix total": sum(nodes["id_prefixes"].values()) == nodes["total_count"],
    "edge schema-group total": sum(row["count"] for row in schema["edges"]) == edges["total_count"],
    "predicate total": sum(edges["predicates"].values()) == edges["total_count"],
}
for name, passed in checks.items():
    print(f"{name}: {'matches' if passed else 'MISMATCH — investigate before interpreting aggregates'}")
assert all(checks.values()), checks


node category-group total: matches
node prefix total: matches
edge schema-group total: matches
predicate total: matches


## 2. Which entities are represented?

Keep exact category combinations together: a node labeled both Gene and Protein
should not be counted as two nodes. These are explicit labels, without Biolink
ancestor expansion. Small molecules are not automatically approved drugs.
Identifier prefixes describe canonical IDs in the release, not all synonyms.


In [3]:
category_counts = Counter()
for row in schema["nodes"]:
    category_counts[" + ".join(sorted(row["category"]))] += row["count"]
category_rows = counts_table(category_counts, "node_categories.csv", nodes["total_count"])
print("\nCanonical identifier prefixes")
prefix_rows = counts_table(nodes["id_prefixes"], "node_prefixes.csv", nodes["total_count"])


label                            | count   | percent
---------------------------------+---------+--------
biolink:SmallMolecule            | 1050118 | 59.84  
biolink:Protein                  | 176208  | 10.04  
biolink:Gene                     | 127322  | 7.26   
biolink:Gene + biolink:Protein   | 124373  | 7.09   
biolink:Disease                  | 51707   | 2.95   
biolink:ChemicalEntity           | 51045   | 2.91   
biolink:PhenotypicFeature        | 39114   | 2.23   
biolink:Pathway                  | 32242   | 1.84   
biolink:BiologicalProcess        | 25036   | 1.43   
biolink:AnatomicalEntity         | 10820   | 0.62   
biolink:MolecularActivity        | 10671   | 0.61   
biolink:MolecularMixture         | 10561   | 0.6    
biolink:GrossAnatomicalStructure | 10471   | 0.6    
biolink:GeneFamily               | 7439    | 0.42   
biolink:CellularComponent        | 5574    | 0.32   
Showing 15 of 42 rows; full table exported.

Canonical identifier prefixes
label            | count

## 3. Which relationships and evidence sources dominate?

A large number of ontology edges can dominate a model's input. We should inspect
their purpose before deciding which to retain. Source counts are source incidences;
do not assume they count distinct biological claims or unique entity pairs.


In [4]:
predicate_rows = counts_table(edges["predicates"], "edge_predicates.csv", edges["total_count"])
print("\nPrimary knowledge sources")
source_rows = counts_table(edges["primary_knowledge_sources"], "edge_sources.csv")
print(f"\nsubclass_of share: {100 * edges['predicates'].get('biolink:subclass_of', 0) / edges['total_count']:.2f}%")


label                                         | count    | percent
----------------------------------------------+----------+--------
biolink:subclass_of                           | 10120364 | 35.25  
biolink:expressed_in                          | 4782771  | 16.66  
biolink:affects                               | 3377726  | 11.77  
biolink:associated_with                       | 2387667  | 8.32   
biolink:directly_physically_interacts_with    | 1230443  | 4.29   
biolink:occurs_in                             | 848572   | 2.96   
biolink:has_participant                       | 664234   | 2.31   
biolink:physically_interacts_with             | 622361   | 2.17   
biolink:located_in                            | 534630   | 1.86   
biolink:treats_or_applied_or_studied_to_treat | 445762   | 1.55   
biolink:in_clinical_trials_for                | 429798   | 1.5    
biolink:genetic_association                   | 415530   | 1.45   
biolink:involved_in                           | 382583   | 1.3

## 4. Treatment-related evidence

This explicit exploratory list keeps treatment, research, and contraindication
predicates separate. It is not an ontology-derived list of descendants and is not
a set of positive training labels. Counts span all category combinations; they
are **edge records**, not unique drug–disease pairs. Contraindications are not
automatically suitable negative labels.


In [5]:
treatment_predicates = {
    "biolink:treats", "biolink:treats_or_applied_or_studied_to_treat",
    "biolink:applied_to_treat", "biolink:studied_to_treat",
    "biolink:in_clinical_trials_for", "biolink:in_preclinical_trials_for",
    "biolink:ameliorates_condition", "biolink:contraindicated_in",
}
treatment_rows = counts_table(
    {p: edges["predicates"].get(p, 0) for p in treatment_predicates},
    "treatment_predicates.csv")
combinations = Counter()
source_pairs = Counter()
for row in schema["edges"]:
    if row["predicate"] in treatment_predicates:
        combinations[(" + ".join(sorted(row["subject_category"])), row["predicate"],
                      " + ".join(sorted(row["object_category"])))] += row["count"]
        for source, count in row.get("primary_knowledge_sources", {}).items():
            source_pairs[(row["predicate"], source)] += count
combination_rows = [dict(zip(["subject_categories", "predicate", "object_categories", "count"], (*key, count)))
                    for key, count in combinations.most_common()]
export("treatment_category_combinations.csv", combination_rows,
       ["subject_categories", "predicate", "object_categories", "count"])
print("\nTreatment-related category combinations")
table(combination_rows, ["subject_categories", "predicate", "object_categories", "count"], limit=12)
treatment_sources = [{"predicate": key[0], "source": key[1], "count": count}
                     for key, count in source_pairs.most_common()]
export("treatment_sources.csv", treatment_sources, ["predicate", "source", "count"])
print("\nSources of explicit biolink:treats edges")
table([row for row in treatment_sources if row["predicate"] == "biolink:treats"],
      ["predicate", "source", "count"])


label                                         | count 
----------------------------------------------+-------
biolink:treats_or_applied_or_studied_to_treat | 445762
biolink:in_clinical_trials_for                | 429798
biolink:applied_to_treat                      | 65254 
biolink:treats                                | 33634 
biolink:contraindicated_in                    | 22859 
biolink:studied_to_treat                      | 1645  
biolink:in_preclinical_trials_for             | 275   
biolink:ameliorates_condition                 | 7     

Treatment-related category combinations
subject_categories                   | predicate                                     | object_categories         | count 
-------------------------------------+-----------------------------------------------+---------------------------+-------
biolink:Drug + biolink:SmallMolecule | biolink:treats_or_applied_or_studied_to_treat | biolink:Disease           | 185178
biolink:Drug + biolink:SmallMolecule | biol

## 5. Qualifiers and metadata coverage

Qualifier fields can carry direction, aspect, species, or other meaning that would
be lost in a naive subject–predicate–object conversion. These field incidences can
overlap; summing them does not give the number of qualified edges. Values and
missing-endpoint/duplicate checks require a later raw-record scan.


In [6]:
qualifiers = Counter()
for row in schema["edges"]:
    qualifiers.update(row.get("qualifiers", {}))
qualifier_rows = counts_table(qualifiers, "qualifier_fields.csv")
print("\nNode attribute coverage")
attribute_rows = counts_table(nodes.get("attributes", {}), "node_attributes.csv", nodes["total_count"])


label                             | count  
----------------------------------+--------
qualified_predicate               | 3302772
object_aspect_qualifier           | 3020478
object_direction_qualifier        | 2893294
species_context_qualifier         | 2169485
frequency_qualifier               | 309426 
causal_mechanism_qualifier        | 307261 
subject_form_or_variant_qualifier | 168185 
disease_context_qualifier         | 156059 
qualifier                         | 127828 
anatomical_context_qualifier      | 122027 
stage_qualifier                   | 121281 
onset_qualifier                   | 3041   
sex_qualifier                     | 587    
object_form_or_variant_qualifier  | 64     

Node attribute coverage
label                    | count   | percent
-------------------------+---------+--------
name                     | 1754754 | 100.0  
equivalent_identifiers   | 1754754 | 100.0  
description              | 605580  | 34.51  
information_content      | 602052  | 34.31  
x

## 6. Archive inventory (streaming, no extraction)

The first run decompresses the archive sequentially to read tar headers; allow a
minute or more. Subsequent runs reuse the small inventory cached in `output/overview`.
Set `REFRESH_INVENTORY = True` after replacing the archive. The manifest fingerprint
identifies the downloaded artifact; this cell does not recompute its checksum.


In [7]:
REFRESH_INVENTORY = False
inventory_path = OUTPUT / "archive_inventory.json"
fingerprint = files["translator_kg.tar.zst"]["sha256"]
inventory = json.loads(inventory_path.read_text()) if inventory_path.exists() else None
if REFRESH_INVENTORY or inventory is None or inventory.get("archive_sha256") != fingerprint:
    if not shutil.which("zstd"):
        raise RuntimeError("Install zstd to inspect the compressed archive.")
    members = []
    with subprocess.Popen(["zstd", "-dc", str(archive)], stdout=subprocess.PIPE,
                          stderr=subprocess.PIPE) as process:
        try:
            with tarfile.open(fileobj=process.stdout, mode="r|") as tar:
                for member in tar:
                    members.append({"name": member.name, "bytes": member.size,
                                    "kind": "file" if member.isfile() else "other"})
            # Drain tar padding so zstd exits normally and verifies the full stream.
            while process.stdout.read(1024 * 1024):
                pass
            error = process.stderr.read().decode()
            if process.wait() != 0:
                raise RuntimeError(error)
        except BaseException:
            process.kill()
            process.wait()
            raise
    inventory = {"archive_sha256": fingerprint, "members": members}
    inventory_path.write_text(json.dumps(inventory, indent=2) + "\n")
table(inventory["members"], ["name", "bytes", "kind"], limit=30)
print(f"\nTotal file payload: {sum(m['bytes'] for m in inventory['members'] if m['kind'] == 'file') / 1e9:.2f} GB")
print("No archive members extracted.")


name                | bytes       | kind
--------------------+-------------+-----
nodes.jsonl         | 1075512212  | file
edges.jsonl         | 24458221015 | file
graph-metadata.json | 3709029     | file

Total file payload: 25.54 GB
No archive members extracted.


## What this means for the next experiment

Use the tables to choose a manageable graph and a precise prediction target.
Before preparing ULTRA inputs, the next analysis should stream raw records to:

- Inspect example nodes, edges, qualifier values, and provenance.
- Define eligible drugs and diseases, including species restrictions where relevant.
- Count unique pairs and duplicate triples; check that edge endpoints exist.
- Measure degree distributions and connectivity with disk-backed or compact structures.
- Separate evaluation pairs before constructing the model graph; remove equivalent
  or inverse answer edges that could leak those labels.

The full graph's size is a reason to benchmark a subset first. Metadata alone does
not establish model memory requirements, valid negative labels, or biomedical
prediction quality. All CSV outputs here are derived from the publisher metadata.
